# Derin Öğrenme ile Görüntü İşleme: CNN, U-Net ve Mask R-CNN

Bu not defteri, görüntü işlemede en çok kullanılan üç derin öğrenme mimarisini
**kavramsal açıklama + satır satır yorumlanmış kod** formatında ele alır:

1. **CNN (Convolutional Neural Network)** — Görüntü sınıflandırma
2. **U-Net** — Piksel bazlı segmentasyon (görüntüdeki her pikseli sınıflandırma)
3. **Mask R-CNN** — Nesne tespiti + örnek (instance) segmentasyonu

> **Not:** Bu defter eğitim amaçlıdır. Küçük/sentetik veri veya `torchvision`
> gibi hazır ağırlıklarla çalışacak şekilde tasarlanmıştır, böylece kod hiçbir
> ek veri seti indirmeden çalıştırılabilir ve mimarilerin nasıl kurulduğu
> görülebilir. Gerçek bir projede kendi veri setinizle eğitim (training)
> döngüsünü genişletmeniz gerekir.

**Kullanılan kütüphane:** PyTorch (`torch`, `torchvision`)


In [ ]:
# Gerekli kütüphaneleri kontrol edip kuruyoruz
# torch: tensör işlemleri ve sinir ağı katmanları için temel kütüphane
# torchvision: hazır modeller (Mask R-CNN dahil) ve görüntü dönüşümleri için
!pip install torch torchvision --quiet


In [ ]:
import torch                     # Ana derin öğrenme kütüphanesi (tensörler, autograd)
import torch.nn as nn            # Sinir ağı katmanlarını içerir (Conv2d, Linear, vb.)
import torch.nn.functional as F  # Aktivasyon fonksiyonları gibi durumsuz (stateless) işlemler
import numpy as np                # Sayısal işlemler ve dizi (array) manipülasyonu için

torch.manual_seed(42)  # Tekrar üretilebilirlik için rastgelelik tohumunu (seed) sabitliyoruz
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU varsa kullan, yoksa CPU
print(f"Kullanılan cihaz: {device}")


---
## 1. CNN (Convolutional Neural Network / Evrişimli Sinir Ağı)

### Konu Anlatımı

CNN, özellikle görüntü verisi için tasarlanmış bir sinir ağı türüdür. Klasik
tam bağlantılı (fully connected) ağların aksine, görüntünün **uzamsal (spatial)
yapısını** korur ve şu üç ana bileşenden oluşur:

1. **Konvolüsyon (Evrişim) Katmanı (`Conv2d`):**
   Küçük bir filtre (kernel), görüntü üzerinde kaydırılarak (sliding window)
   yerel örüntüleri (kenar, köşe, doku vb.) tespit eder. Her filtre bir
   "özellik haritası" (feature map) üretir.

2. **Havuzlama (Pooling) Katmanı (`MaxPool2d`):**
   Özellik haritalarının boyutunu küçültür, hesaplama yükünü azaltır ve
   modeli küçük kaymalara/gürültüye karşı daha dayanıklı (invariant) yapar.

3. **Aktivasyon Fonksiyonu (`ReLU`):**
   Doğrusal olmayanlık (non-linearity) katar; bu sayede ağ karmaşık örüntüleri
   öğrenebilir.

Ağın sonunda genellikle **tam bağlantılı (fully connected / Linear) katmanlar**
bulunur ve bunlar çıkarılan özellikleri sınıf olasılıklarına dönüştürür.

**Tipik kullanım alanı:** Görüntü sınıflandırma (bu görüntü kedi mi köpek mi?).

Aşağıda, MNIST benzeri 28x28 gri tonlamalı görüntüleri 10 sınıfa ayıran basit
bir CNN mimarisi kuruyoruz.


In [ ]:
class BasitCNN(nn.Module):
    """
    Basit bir görüntü sınıflandırma CNN'i.
    Giriş: (batch, 1, 28, 28) boyutunda gri tonlamalı görüntü
    Çıkış: (batch, 10) boyutunda sınıf skorları (logits)
    """
    def __init__(self, num_classes=10):
        super().__init__()  # nn.Module'ün kurucusunu çağırıyoruz (zorunlu)

        # --- 1. Konvolüsyon Bloğu ---
        # in_channels=1: gri tonlamalı görüntü (RGB olsaydı 3 olurdu)
        # out_channels=16: 16 farklı filtre öğrenilecek, yani 16 özellik haritası üretilecek
        # kernel_size=3: 3x3 boyutunda filtre
        # padding=1: kenarlarda bilgi kaybını önlemek için görüntü etrafına 1 piksel sıfır ekle
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)  # Eğitimi stabilize eden normalizasyon katmanı

        # --- 2. Konvolüsyon Bloğu ---
        # Bir önceki katmanın çıktısı (16 kanal) burada girdi olur, 32 kanala çıkarılır
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        # Havuzlama: her 2x2'lik alanın maksimumunu alarak boyutu yarıya indirir
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # İki kez pool uygulanacağı için: 28 -> 14 -> 7
        # Son özellik haritası boyutu: 32 kanal x 7 x 7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)   # Tam bağlantılı katman: özellikleri 128 boyuta indirir
        self.dropout = nn.Dropout(0.3)          # Aşırı öğrenmeyi (overfitting) azaltmak için nöronların %30'unu rastgele söndürür
        self.fc2 = nn.Linear(128, num_classes)  # Son katman: her sınıf için bir skor üretir

    def forward(self, x):
        # x boyutu: (batch, 1, 28, 28)

        x = self.conv1(x)          # Konvolüsyon uygula -> (batch, 16, 28, 28)
        x = self.bn1(x)            # Batch normalization uygula
        x = F.relu(x)              # Negatif değerleri 0 yapan aktivasyon fonksiyonu
        x = self.pool(x)           # Boyutu yarıya indir -> (batch, 16, 14, 14)

        x = self.conv2(x)          # İkinci konvolüsyon -> (batch, 32, 14, 14)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool(x)           # -> (batch, 32, 7, 7)

        x = torch.flatten(x, 1)    # 2D özellik haritalarını 1D vektöre düzleştir -> (batch, 32*7*7)
        x = self.fc1(x)            # Tam bağlantılı katmandan geçir -> (batch, 128)
        x = F.relu(x)
        x = self.dropout(x)        # Dropout uygula (sadece eğitim sırasında aktif)
        x = self.fc2(x)            # Son katman -> (batch, num_classes) ham skorlar (logits)

        return x


In [ ]:
# Modeli oluşturup cihaza (CPU/GPU) taşıyoruz
model_cnn = BasitCNN(num_classes=10).to(device)
print(model_cnn)

# Sahte (dummy) bir görüntü batch'i ile ileri yayılımı (forward pass) test edelim
# 8 adet, 1 kanallı, 28x28 boyutunda rastgele görüntü
sahte_girdi = torch.randn(8, 1, 28, 28).to(device)
cikti = model_cnn(sahte_girdi)

print(f"Girdi boyutu : {sahte_girdi.shape}")
print(f"Çıktı boyutu : {cikti.shape}  # (batch_size=8, num_classes=10)")


**Eğitim döngüsü (özet):** Gerçek bir eğitimde şu adımlar döngü içinde
tekrarlanır: (1) veriyi modele ver, (2) kayıp (loss) hesapla — genellikle
`CrossEntropyLoss`, (3) geri yayılım (`loss.backward()`) ile gradyanları
hesapla, (4) optimizer (`Adam`, `SGD` vb.) ile ağırlıkları güncelle. Aşağıda
bu döngünün iskeleti (tek adım) gösterilmiştir.

In [ ]:
# Kayıp fonksiyonu: çok sınıflı sınıflandırma için standart seçim
criterion = nn.CrossEntropyLoss()

# Optimizer: ağırlıkları gradyanlara göre güncelleyen algoritma
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

# --- Tek bir eğitim adımı örneği (sahte veriyle) ---
sahte_etiketler = torch.randint(0, 10, (8,)).to(device)  # 8 örnek için rastgele doğru sınıflar

optimizer.zero_grad()                    # Önceki adımdan kalan gradyanları sıfırla
cikti = model_cnn(sahte_girdi)           # İleri yayılım: tahminleri hesapla
loss = criterion(cikti, sahte_etiketler) # Tahmin ile gerçek etiket arasındaki hatayı hesapla
loss.backward()                          # Geri yayılım: her ağırlık için gradyanı hesapla
optimizer.step()                         # Ağırlıkları gradyana göre güncelle

print(f"Örnek eğitim adımı kaybı (loss): {loss.item():.4f}")


---
## 2. U-Net

### Konu Anlatımı

U-Net, orijinal olarak tıbbi görüntü segmentasyonu için geliştirilmiş, ancak
günümüzde her türlü **piksel bazlı segmentasyon** görevinde kullanılan bir
mimaridir. Adını, ağın "U" harfine benzeyen simetrik yapısından alır.

İki ana yoldan oluşur:

1. **Daraltma Yolu (Encoder / Contracting Path):**
   Klasik bir CNN gibi çalışır: art arda konvolüsyon + havuzlama katmanları
   ile görüntünün **uzamsal boyutu küçülür** ama **kanal (özellik) sayısı
   artar**. Bu yol "ne var" sorusuna cevap arar (bağlamsal/semantic bilgi).

2. **Genişletme Yolu (Decoder / Expanding Path):**
   Ters işlem uygulanır: transpoze konvolüsyon (upsampling) ile uzamsal
   boyut tekrar büyütülür. Bu yol "nerede" sorusuna cevap arar.

3. **Atlama Bağlantıları (Skip Connections):**
   U-Net'in en kritik özelliği budur. Encoder'daki her seviyenin çıktısı,
   decoder'daki aynı seviyeye **doğrudan bağlanır (concatenate)**. Bu sayede
   havuzlama sırasında kaybolan ince mekansal detaylar (nesne sınırları gibi)
   decoder'a geri taşınır ve segmentasyon çok daha keskin sınırlarla yapılır.

**Çıktı:** Girdiyle aynı boyutta (H x W), her pikselin hangi sınıfa ait
olduğunu gösteren bir "maske" (segmentasyon haritası).

**Tipik kullanım alanı:** Tümör/organ segmentasyonu, uydu görüntülerinde arazi
sınıflandırma, otonom araçlarda yol/şerit segmentasyonu.


In [ ]:
class ConvBlok(nn.Module):
    """
    U-Net'te tekrar tekrar kullanılan temel yapı taşı:
    (Konvolüsyon -> BatchNorm -> ReLU) x 2
    Bu blok hem encoder hem decoder tarafında kullanılır.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.blok = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),  # Uzamsal boyutu koru (padding=1)
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),  # İkinci konvolüsyon, kanal sayısı sabit kalır
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.blok(x)  # Sequential içindeki tüm katmanları sırayla uygula


In [ ]:
class UNet(nn.Module):
    """
    Basitleştirilmiş U-Net mimarisi.
    Giriş : (batch, in_ch, H, W)  -- örn. RGB görüntü (in_ch=3)
    Çıkış : (batch, num_classes, H, W) -- her piksel için sınıf skorları
    """
    def __init__(self, in_ch=3, num_classes=1, ozellikler=[64, 128, 256, 512]):
        super().__init__()

        self.encoder = nn.ModuleList()  # Daraltma yolundaki blokları tutacak liste
        self.decoder = nn.ModuleList()  # Genişletme yolundaki blokları tutacak liste
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Her encoder adımından sonra boyutu yarıya indirir

        # --- ENCODER (Daraltma Yolu) oluşturuluyor ---
        kanal = in_ch
        for f in ozellikler:
            self.encoder.append(ConvBlok(kanal, f))  # örn: 3->64, 64->128, 128->256, 256->512
            kanal = f

        # --- BOTTLENECK (En dar/derin katman, encoder ile decoder arasındaki köprü) ---
        self.bottleneck = ConvBlok(ozellikler[-1], ozellikler[-1] * 2)  # 512 -> 1024

        # --- DECODER (Genişletme Yolu) oluşturuluyor ---
        # Ozellikler listesini ters çeviriyoruz çünkü decoder küçükten büyüğe değil,
        # derinden (dar) sığa (geniş) doğru ilerler
        for f in reversed(ozellikler):
            # Transpoze konvolüsyon: özellik haritasının boyutunu 2 katına çıkarır (upsampling)
            self.decoder.append(
                nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2)
            )
            # Skip connection ile birleştirildikten sonra kanal sayısı 2 katına çıkar (f*2),
            # bu yüzden ConvBlok girişi f*2 olarak ayarlanır
            self.decoder.append(ConvBlok(f * 2, f))

        # Son katman: kanal sayısını istenen sınıf sayısına indirir (1x1 konvolüsyon)
        self.son_katman = nn.Conv2d(ozellikler[0], num_classes, kernel_size=1)

    def forward(self, x):
        skip_baglantilari = []  # Her encoder seviyesinin çıktısını burada saklayacağız

        # --- Encoder'dan geçiş ---
        for enc_blok in self.encoder:
            x = enc_blok(x)                # Konvolüsyon bloğunu uygula
            skip_baglantilari.append(x)    # Havuzlamadan ÖNCEKİ hali sakla (decoder'a taşınacak)
            x = self.pool(x)               # Boyutu yarıya indir

        # --- Bottleneck ---
        x = self.bottleneck(x)  # En derin katman

        # Skip bağlantılarını decoder'da kullanmak için ters çeviriyoruz
        # (en son eklenen encoder çıktısı, decoder'da en önce kullanılır)
        skip_baglantilari = skip_baglantilari[::-1]

        # --- Decoder'dan geçiş ---
        # self.decoder listesinde ikişerli gidiyoruz: [upsample, convblok, upsample, convblok, ...]
        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x)                    # Transpoze konvolüsyon ile boyutu büyüt
            skip = skip_baglantilari[i // 2]           # İlgili encoder seviyesinin sakladığı özellik haritası

            # Boyutlar (yuvarlama hatalarından dolayı) tam eşleşmeyebilir, gerekirse kırp
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])  # skip ile aynı H,W boyutuna getir

            x = torch.cat((skip, x), dim=1)           # KRİTİK ADIM: skip connection -> kanal boyutunda birleştir
            x = self.decoder[i + 1](x)                # ConvBlok ile işle

        return self.son_katman(x)  # Son 1x1 konvolüsyon ile sınıf haritasını üret


In [ ]:
# U-Net modelini oluşturuyoruz: 3 kanallı (RGB) girdi, 1 sınıflı çıktı (örn. ikili segmentasyon: nesne/arkaplan)
model_unet = UNet(in_ch=3, num_classes=1).to(device)

# Sahte bir RGB görüntü batch'i: (batch=2, kanal=3, yükseklik=160, genişlik=160)
sahte_goruntu = torch.randn(2, 3, 160, 160).to(device)
maske_ciktisi = model_unet(sahte_goruntu)

print(f"Girdi boyutu : {sahte_goruntu.shape}")
print(f"Çıktı boyutu : {maske_ciktisi.shape}  # Girdiyle aynı H,W -> piksel bazlı segmentasyon haritası")

# Toplam öğrenilebilir parametre sayısı
toplam_param = sum(p.numel() for p in model_unet.parameters())
print(f"Toplam parametre sayısı: {toplam_param:,}")


**Kayıp fonksiyonu notu:** Segmentasyon görevlerinde genellikle
`BCEWithLogitsLoss` (ikili segmentasyon için) veya `CrossEntropyLoss`
(çok sınıflı segmentasyon için) kullanılır; ayrıca sınırların örtüşme
kalitesini ölçen **Dice Loss** de sıkça bu kayıplarla birlikte kullanılır.

---
## 3. Mask R-CNN

### Konu Anlatımı

Mask R-CNN, **nesne tespiti (object detection)** ile **örnek segmentasyonunu
(instance segmentation)** aynı anda yapan bir mimaridir. U-Net'ten farkı:
U-Net görüntüdeki "kedi bölgesi" gibi bir sınıf maskesi üretirken, Mask R-CNN
her bir **nesne örneğini ayrı ayrı** tespit eder (örn. görüntüdeki 3 farklı
köpeği birbirinden ayırt eder) ve her biri için hem sınırlayıcı kutu
(bounding box) hem de o nesneye özel bir maske üretir.

Mimarinin temel bileşenleri:

1. **Omurga Ağı (Backbone, örn. ResNet + FPN):**
   Görüntüden çok ölçekli özellik haritaları çıkarır.

2. **Bölge Öneri Ağı (Region Proposal Network - RPN):**
   Görüntüde nesne olabilecek aday bölgeleri (region proposals) hızlıca
   önerir.

3. **RoIAlign:**
   Faster R-CNN'deki RoIPooling'in daha hassas bir versiyonudur; aday
   bölgeleri sabit boyutlu özellik haritalarına dönüştürürken piksel
   hizalama hatalarını önler (maske kalitesi için kritiktir).

4. **Üç Paralel Çıkış Başlığı (Head):**
   - **Sınıflandırma başlığı:** Nesnenin hangi sınıfa ait olduğunu tahmin eder
   - **Kutu regresyon başlığı:** Sınırlayıcı kutunun koordinatlarını hassaslaştırır
   - **Maske başlığı:** Küçük bir FCN (Fully Convolutional Network) ile o
     nesneye özel ikili (binary) piksel maskesini üretir

**Tipik kullanım alanı:** Otonom sürüş (araç/yaya sayma ve sınırlama),
video izleme, tıbbi görüntülerde çoklu hücre/lezyon tespiti.

Aşağıda, sıfırdan mimari kurmak yerine (bu çok karmaşık olur),
`torchvision`'ın hazır Mask R-CNN mimarisini kullanıp **kendi sınıf
sayımıza göre uyarlıyoruz** — gerçek projelerde de yaygın yaklaşım budur
(transfer learning).


In [ ]:
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def mask_rcnn_olustur(num_classes):
    """
    COCO veri setiyle önceden eğitilmiş (pretrained) Mask R-CNN modelini yükler
    ve son katmanlarını kendi sınıf sayımıza göre değiştirir (transfer learning).

    num_classes: arkaplan dahil toplam sınıf sayısı (örn. 2 = arkaplan + 1 nesne sınıfı)
    """
    # weights="DEFAULT": ImageNet/COCO üzerinde önceden eğitilmiş ağırlıkları indirir
    # Bu sayede sıfırdan eğitmek yerine, öğrenilmiş genel görsel özelliklerden faydalanırız
    model = maskrcnn_resnet50_fpn(weights="DEFAULT")

    # --- 1. Sınıflandırma + Kutu Regresyon Başlığını Değiştir ---
    in_features = model.roi_heads.box_predictor.cls_score.in_features  # Mevcut özellik boyutunu al
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # --- 2. Maske Başlığını Değiştir ---
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256  # Maske başlığındaki gizli katman boyutu
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model


In [ ]:
# Örnek: arkaplan + 2 nesne sınıfı (toplam 3 sınıf) için model oluşturuyoruz
NUM_CLASSES = 3  # örn: 0=arkaplan, 1=kedi, 2=köpek
model_maskrcnn = mask_rcnn_olustur(NUM_CLASSES).to(device)
model_maskrcnn.eval()  # Değerlendirme (inference) moduna al (BatchNorm/Dropout davranışını sabitler)

print("Mask R-CNN modeli başarıyla oluşturuldu.")
print(f"Toplam parametre sayısı: {sum(p.numel() for p in model_maskrcnn.parameters()):,}")


In [ ]:
# --- Çıkarım (Inference) Örneği ---
# Mask R-CNN, girdi olarak [0,1] aralığında normalize edilmiş RGB görüntü listesi bekler
# (sabit boyut şartı yoktur, model içeride kendi ölçekler)

sahte_goruntu_listesi = [torch.rand(3, 300, 400).to(device)]  # 1 adet sahte RGB görüntü

with torch.no_grad():  # Çıkarım sırasında gradyan hesaplamayı kapatıyoruz (bellek/hız için)
    tahminler = model_maskrcnn(sahte_goruntu_listesi)

# Model her görüntü için bir sözlük (dict) döndürür:
# 'boxes'  -> tespit edilen her nesnenin sınırlayıcı kutu koordinatları [x1,y1,x2,y2]
# 'labels' -> her kutunun tahmin edilen sınıf etiketi
# 'scores' -> her tahminin güven skoru (0-1 arası)
# 'masks'  -> her nesne için piksel bazlı ikili maske (1, H, W) boyutunda

tek_goruntu_sonucu = tahminler[0]
print("Çıktı anahtarları:", tek_goruntu_sonucu.keys())
print(f"Tespit edilen kutu sayısı : {tek_goruntu_sonucu['boxes'].shape}")
print(f"Maske tensör boyutu       : {tek_goruntu_sonucu['masks'].shape}")
print(f"Skor örnekleri (ilk 5)    : {tek_goruntu_sonucu['scores'][:5]}")


**Eğitim notu:** Mask R-CNN'i kendi verinizle eğitmek için her görüntüye
karşılık `boxes`, `labels` ve `masks` içeren hedef (target) sözlükleri
hazırlamanız ve `torchvision.datasets` formatına uygun bir `Dataset` sınıfı
yazmanız gerekir. Model eğitim modundayken (`model.train()`) hem sınıflandırma
hem kutu hem de maske kayıplarını otomatik olarak birlikte hesaplar ve
`loss_dict` olarak döner; bu kayıpların toplamı geri yayılım için kullanılır.


---
## Özet Karşılaştırma

| Özellik | CNN | U-Net | Mask R-CNN |
|---|---|---|---|
| **Görev** | Sınıflandırma | Semantik segmentasyon | Nesne tespiti + örnek segmentasyonu |
| **Çıktı** | Tek etiket/görüntü | Piksel bazlı sınıf haritası | Kutu + sınıf + nesne bazlı maske |
| **Nesneleri ayırt eder mi?** | Hayır | Hayır (aynı sınıf tek maskede birleşir) | Evet (her nesne ayrı) |
| **Mimari karmaşıklığı** | Düşük | Orta | Yüksek |
| **Tipik kullanım** | "Bu görüntü ne?" | "Hangi piksel hangi sınıfa ait?" | "Kaç nesne var, nerede, sınırları ne?" |

### Sonraki Adımlar
- Gerçek bir veri seti ile (örn. `torchvision.datasets.MNIST`, Kaggle segmentasyon setleri, COCO) eğitim döngüsünü tamamlayabilirsiniz.
- `DataLoader` kullanarak batch'ler halinde veri besleme eklenebilir.
- Eğitim/doğrulama kaybı grafikleri (`matplotlib`) ile izlenebilir.
- Model ağırlıkları `torch.save(model.state_dict(), "model.pth")` ile kaydedilebilir.
